# csv module
- **CSV** (Comma-Separated Values) is the most common format for tabular data — every row is a line, values separated by `,`.
- The `csv` module reads/writes CSV files correctly (handles quotes, commas inside values, etc.). Don't just do `line.split(",")`!

### Most used tools ⭐
| Tool | Each row is a... |
|---|---|
| `csv.reader(f)` | `list` → `['Ali', '22']` |
| `csv.writer(f)` | write lists with `.writerow()` / `.writerows()` |
| `csv.DictReader(f)` | `dict` → `{'name': 'Ali', 'age': '22'}` (uses header row as keys) |
| `csv.DictWriter(f, fieldnames)` | write dicts, `.writeheader()` first |

### Rules to remember
1. Open files with **`newline=""`** — otherwise you get empty lines between rows on Windows.
2. Everything you read is a **string** → convert with `int()` / `float()`.
3. Other separators: `delimiter=";"` or `delimiter="\t"` (TSV).
4. For big data analysis you'll use **pandas** (`pd.read_csv`), but `csv` is built-in and great for simple tasks.

In [ ]:
import csv

### 1. `csv.writer` — write rows as lists

In [ ]:
rows = [
    ["name", "age", "city"],
    ["Ali", 22, "Cairo"],
    ["Sara", 31, "Alex, Egypt"],    # value contains a comma -> csv adds quotes automatically
]
with open("people.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(rows[0])     # one row
    writer.writerows(rows[1:])   # many rows

print(open("people.csv").read())

### 2. `csv.reader` — read rows as lists
Use `next(reader)` to grab the header row.

In [ ]:
with open("people.csv", newline="") as f:
    reader = csv.reader(f)
    header = next(reader)
    print("header:", header)
    for row in reader:
        print(row, "-> age as int:", int(row[1]))

### 3. `csv.DictWriter` — write rows as dicts ⭐

In [ ]:
people = [
    {"name": "Omar", "age": 27, "city": "Giza"},
    {"name": "Mona", "age": 19, "city": "Cairo"},
]
with open("people_dict.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["name", "age", "city"])
    writer.writeheader()
    writer.writerows(people)

print(open("people_dict.csv").read())

### 4. `csv.DictReader` — read rows as dicts ⭐
Most readable way: access by column name instead of index.

In [ ]:
with open("people_dict.csv", newline="") as f:
    for row in csv.DictReader(f):
        print(row["name"], int(row["age"]))

In [ ]:
# common pattern: load whole file into a list of dicts
with open("people_dict.csv", newline="") as f:
    data = list(csv.DictReader(f))
print(data)
print(len(data), "rows")

### 5. Other delimiters

In [ ]:
with open("semi.csv", "w", newline="") as f:
    csv.writer(f, delimiter=";").writerows([["a", "b"], [1, 2]])
print(open("semi.csv").read())

with open("semi.csv", newline="") as f:
    print(list(csv.reader(f, delimiter=";")))

In [ ]:
# cleanup
import os
for name in ["people.csv", "people_dict.csv", "semi.csv"]:
    os.remove(name)

## 🏋️ Exercises
- Try each exercise yourself in the empty cell first, then open **💡 Show solution** to compare.
- Run the **setup** cell first.
- If something breaks, just re-run the setup cell to reset everything.

In [ ]:
# SETUP: creates a practice folder (re-run anytime to reset)
import csv, json, shutil
from pathlib import Path

base = Path("ex_csv")
shutil.rmtree(base, ignore_errors=True)
base.mkdir()

(base / "students.csv").write_text(
"""name,age,grade,city
Ali,22,88,Cairo
Sara,21,95,Alex
Omar,23,67,Cairo
Mona,20,74,Giza
Youssef,22,81,Alex
Nour,21,59,Cairo
""")

(base / "sales.csv").write_text(
"""date,product,qty,price
2026-11-01,laptop,2,15000
2026-11-01,mouse,10,200
2026-11-02,laptop,1,15000
2026-11-02,keyboard,5,600
2026-11-03,mouse,4,200
""")

(base / "europe.csv").write_text(
"""city;temp
Berlin;4,5
Paris;7,2
""")
print("setup done:", sorted(p.name for p in base.iterdir()))

### Exercise 1
**Read basics** — read `ex_csv/students.csv` with `csv.reader`. Print the header and the number of data rows (without the header).

In [ ]:
# your code here

<details>
<summary>💡 Show solution</summary>

```python
with open(base / "students.csv", newline="") as f:
    reader = csv.reader(f)
    header = next(reader)
    rows = list(reader)
print(header)
print(len(rows))
```
</details>

### Exercise 2
**Average grade** — using `csv.DictReader`, compute the average grade and print the name of the top student.
Remember: values are strings!

In [ ]:
# your code here

<details>
<summary>💡 Show solution</summary>

```python
with open(base / "students.csv", newline="") as f:
    students = list(csv.DictReader(f))

grades = [int(s["grade"]) for s in students]
print(sum(grades) / len(grades))
print(max(students, key=lambda s: int(s["grade"]))["name"])
```
</details>

### Exercise 3
**Filter & write** — write all students with `grade >= 75` to `ex_csv/passed.csv` (same columns, with header). Print the new file.

In [ ]:
# your code here

<details>
<summary>💡 Show solution</summary>

```python
with open(base / "students.csv", newline="") as f:
    reader = csv.DictReader(f)
    passed = [row for row in reader if int(row["grade"]) >= 75]
    fieldnames = reader.fieldnames

with open(base / "passed.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(passed)

print((base / "passed.csv").read_text())
```
</details>

### Exercise 4
**Total sales per product** — from `ex_csv/sales.csv`, build a dict `{product: total money}` where money = `qty * price`.
Expected: `{'laptop': 45000, 'mouse': 2800, 'keyboard': 3000}`

In [ ]:
# your code here

<details>
<summary>💡 Show solution</summary>

```python
totals = {}
with open(base / "sales.csv", newline="") as f:
    for row in csv.DictReader(f):
        money = int(row["qty"]) * int(row["price"])
        totals[row["product"]] = totals.get(row["product"], 0) + money
print(totals)
```
</details>

### Exercise 5
**Add a column** — create `ex_csv/students_status.csv` with all columns of `students.csv` plus a new `status` column: `"pass"` if grade >= 60, else `"fail"`.

In [ ]:
# your code here

<details>
<summary>💡 Show solution</summary>

```python
with open(base / "students.csv", newline="") as f:
    reader = csv.DictReader(f)
    rows = list(reader)
    fieldnames = reader.fieldnames + ["status"]

for row in rows:
    row["status"] = "pass" if int(row["grade"]) >= 60 else "fail"

with open(base / "students_status.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(rows)

print((base / "students_status.csv").read_text())
```
</details>

### Exercise 6
**CSV → JSON** — convert `students.csv` to `ex_csv/students.json` as a list of dicts, where `age` and `grade` are **numbers**, not strings.

In [ ]:
# your code here

<details>
<summary>💡 Show solution</summary>

```python
with open(base / "students.csv", newline="") as f:
    students = list(csv.DictReader(f))

for s in students:
    s["age"] = int(s["age"])
    s["grade"] = int(s["grade"])

with open(base / "students.json", "w") as f:
    json.dump(students, f, indent=4)

print(json.loads((base / "students.json").read_text())[0])
```
</details>

### Exercise 7
**Different delimiter** — `ex_csv/europe.csv` uses `;` as separator and `,` as the decimal point (`4,5` means 4.5).
Read it and print each city with its temperature as a `float`.

In [ ]:
# your code here

<details>
<summary>💡 Show solution</summary>

```python
with open(base / "europe.csv", newline="") as f:
    for row in csv.DictReader(f, delimiter=";"):
        print(row["city"], float(row["temp"].replace(",", ".")))
```
</details>

### Cleanup
Run this when you're done to delete the practice folder.

In [ ]:
shutil.rmtree("ex_csv", ignore_errors=True)
print(Path("ex_csv").exists())